# SOC Security Framework — Demo

Demonstrates the contrast between the unprotected pipeline and the secured pipeline.
The framework runs persistently in `framework-server:8090`.
This notebook only sends HTTP requests.

In [ ]:
import os, requests, json
from IPython.display import display, Markdown

BASE = os.getenv('FRAMEWORK_SERVER_URL', 'http://localhost:8090')

def pp(data):
    print(json.dumps(data, indent=2))

## 1. Framework status

In [ ]:
status = requests.get(f'{BASE}/status').json()
pp(status)

## 2. Registered tools

In [ ]:
tools = requests.get(f'{BASE}/tools').json()
print(f"Wazuh MCP: {len(tools['wazuh'])} tools")
print(f"Evil MCP:  {len(tools['evil'])} tools (after registration validator)")
print()
print('Wazuh tools:', tools['wazuh'])
print('Evil tools (clean):', tools['evil'])

## 3. Run pipeline (latest Wazuh alert)

In [ ]:
result = requests.post(f'{BASE}/pipeline', json={}).json()
print(json.dumps(result, indent=2))

print('=== ALERT ===')
print(result['alert'][:300])
print()
print('=== TRIAGE ===')
pp(result['triage'])
print()
print('=== ENRICHMENT ===')
pp(result['enrichment'])
print()
print('=== RESPONSE ===')
pp(result['response'])

## 4. Audit summary

In [ ]:
summary = requests.get(f'{BASE}/audit/summary').json()
pp(summary)

## 5. Custom alert

In [ ]:
alert = """Latest alert
Alert ID: TEST-001
Time: 2026-06-10T10:00:00.000+0000
Agent: node1
Level: 10
Description: SSH brute force detected: 50 failed login attempts from 10.0.0.1
Rule ID: 5763"""

result = requests.post(f'{BASE}/pipeline', json={'alert': alert}).json()
pp(result['triage'])
pp(result['response'])

## 6. Framework attack demo — access control

In [ ]:
# Triage agent attempting to call propose_wazuh_rule (response_agent only)
r = requests.post(f'{BASE}/call_tool', json={
    'agent_role': 'triage_agent',
    'tool_name':  'propose_wazuh_rule',
    'arguments':  {'rule_id': 99999, 'rule_xml': '<rule id="99999"/>'}
}).json()

print('Blocked:', r['blocked'])
print('Layer:  ', r.get('layer'))
print('Reason: ', r.get('reason'))

## 7. Reset session

In [ ]:
reset = requests.post(f'{BASE}/session/reset').json()
print('New session:', reset['session_id'])